In [2]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd
import glob

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

from datetime import datetime

yyyymmdd = datetime.today().strftime("%Y%m%d")



In [4]:
pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_NEW').to_csv('REMM_Parcels_2023.csv')


In [45]:
outputs = [f'.\\Outputs\\SE_v10_{yyyymmdd}', "SE.gdb"]

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])


In [46]:
# Inputs
se_folder = r'F:\SHARED\Andy\_temp\REMM\02022026\REMM_6Runs_AVG'
temp_se_for_income_and_enrollment = pd.read_csv(r"..\Inputs\SE\SE_2023_6Runs_TAZ920_20251217.csv")
taz9_shp = pd.DataFrame.spatial.from_featureclass(r"..\Inputs\Boundaries\TAZ_900.shp")
taz10_shp = r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Boundaries\WFv910_TAZ_MAG_Update.shp"
temp_box_elder_folder = r'F:\SHARED\Andy\_temp\REMM\02022026\BoxElder'

# select years
start_yr = 2023
end_yr = 2055

In [47]:
print(taz9_shp.columns)

Index(['FID', 'TAZID', 'TAZID_V832', 'SORT', 'CO_IDX', 'CO_TAZID', 'SUBAREAID',
       'ACRES', 'DEVACRES', 'DEVPBLEPCT', 'X', 'Y', 'ADJ_XY', 'CO_FIPS',
       'CO_NAME', 'CITY_FIPS', 'CITY_UGRC', 'CITY_NAME', 'DISTSUPER',
       'DSUP_NAME', 'DISTLRG', 'DLRG_NAME', 'DISTMED', 'DMED_NAME', 'DISTSML',
       'DSML_NAME', 'CBD', 'TERMTIME', 'PRKCSTPERM', 'PRKCSTTEMP', 'WALK100',
       'ECOEDPASS', 'FREEFARE', 'REMM', 'SHAPE'],
      dtype='object')


In [48]:
print(pd.read_csv(r"..\Inputs\SE\SE_2023_6Runs_TAZ900_20260202.csv").columns)

Index([';TAZID', 'CO_TAZID', 'CO_FIPS', 'CO_NAME', 'TOTHH', 'HHPOP', 'HHSIZE',
       'TOTEMP', 'RETEMP', 'INDEMP', 'OTHEMP', 'ALLEMP', 'RETL', 'FOOD',
       'MANU', 'WSLE', 'OFFI', 'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING',
       'FM_CONS', 'HBJ', 'AVGINCOME', 'Enrol_Elem', 'Enrol_Midl',
       'Enrol_High'],
      dtype='object')


In [49]:
# get income and enrollment, for now we will use same data for every year
temp_se_for_income_and_enrollment = temp_se_for_income_and_enrollment[[';TAZID', 'AVGINCOME','Enrol_Elem','Enrol_Midl','Enrol_High']].copy()
temp_se_for_income_and_enrollment.rename({';TAZID':'TAZID'}, axis=1, inplace=True)

In [50]:
cols_to_apportion = ['TOTHH', 'HHPOP', 'TOTEMP', 'RETEMP', 'INDEMP', 'OTHEMP', 'ALLEMP', 'RETL', 'FOOD',
                     'MANU', 'WSLE', 'OFFI', 'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 'HBJ'] 

for yr in range(start_yr,end_yr + 1):
    
    if yr % 5 == 0 or yr in [start_yr,end_yr]:
        print(f'working on {yr}')

    # read in se csv and remove box elder data
    se_df = pd.read_csv(os.path.join(se_folder, f'SE_{yr}.csv'))
    se_df.rename({';TAZID':'TAZID'}, axis=1, inplace=True)
    se_df_no_boxelder = se_df[se_df['CO_FIPS'] != 3].copy()
    se_df_no_boxelder = se_df_no_boxelder[['TAZID'] + cols_to_apportion].copy()

    # read in temporary box elder data
    boxelder_se_df = pd.read_csv(os.path.join(temp_box_elder_folder, f'SE_{yr}.csv'))
    boxelder_se_df.rename({';TAZID':'TAZID'}, axis=1, inplace=True)
    boxelder_se_df = boxelder_se_df[['TAZID'] + cols_to_apportion].copy()

    # add new box elder data
    se_df = pd.concat([boxelder_se_df, se_df_no_boxelder])

    # merge to TAZ9 shape and export
    se_sdf = taz9_shp[['TAZID', 'SHAPE', 'CO_TAZID', 'CO_FIPS', 'CO_NAME']].merge(se_df, on='TAZID', how='left')
    se_fc = os.path.join(gdb, f'SE_{yr}_v9')
    se_sdf.spatial.to_featureclass(location=se_fc,sanitize_columns=False)

    # apportion to TAZ10
    se_fc_v10 = os.path.join(gdb, f'SE_{yr}_v10')

    arcpy.analysis.ApportionPolygon(
        in_features=os.path.realpath(se_fc),
        apportion_fields="TOTHH SUM;HHPOP SUM;TOTEMP SUM;RETEMP SUM;INDEMP SUM;OTHEMP SUM;ALLEMP SUM;RETL SUM;FOOD SUM;MANU SUM;WSLE SUM;OFFI SUM;GVED SUM;HLTH SUM;OTHR SUM;FM_AGRI SUM;FM_MING SUM;FM_CONS SUM;HBJ SUM",
        target_features=taz10_shp,
        out_features=se_fc_v10,
        method="AREA",
        estimation_features=None,
        weight_field=None,
        maintain_geometries="MAINTAIN_GEOMETRIES"
    )

    # read result into sdf
    se_taz10_sdf = pd.DataFrame.spatial.from_featureclass(se_fc_v10)

    # calc household size
    se_taz10_sdf['HHSIZE'] = np.where(se_taz10_sdf["HHPOP"] != 0, se_taz10_sdf["TOTHH"] / se_taz10_sdf["HHPOP"], 0)

    # add enrollment and income
    se_taz10_sdf = se_taz10_sdf.merge(temp_se_for_income_and_enrollment, on='TAZID', how='left')

    # reorder columns
    se_taz10_sdf = se_taz10_sdf[['TAZID', 'CO_TAZID','TOTHH', 'HHPOP', 'HHSIZE', 
                                'TOTEMP', 'RETEMP', 'INDEMP', 'OTHEMP', 'ALLEMP', 
                                'RETL', 'FOOD', 'MANU', 'WSLE', 'OFFI', 'GVED', 
                                'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 
                                'HBJ','AVGINCOME','Enrol_Elem','Enrol_Midl',
                                'Enrol_High', 'CO_FIPS', 'CO_NAME']].copy()

    # export to csv
    se_df_for_csv = se_taz10_sdf.rename({'TAZID':';TAZID'}, axis=1)
    se_df_for_csv.to_csv(os.path.join(outputs[0], f'SE_{yr}.csv'))

working on 2023
working on 2025
working on 2030
working on 2035
working on 2040
working on 2045
working on 2050
working on 2055
